# Generacion de datos - Tesis MAFI USACH

Este notebook permite ejecutar paso a paso la simulacion de datos para la
tesis de resiliencia financiera.

La logica base vive en `scripts/generate_resilience_simulation.py`, para
mantener una sola fuente de verdad.


## 1. Preparar entorno

Ejecuta esta celda primero. Define rutas e importa las funciones del generador.


In [1]:
from pathlib import Path
from random import Random
import sys

import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR))

from scripts import generate_resilience_simulation as sim

PROJECT_DIR


PosixPath('/Users/alejandroalvear/Documents/Alejandro/Desarrollo/Finanzas/MachineLearning/SAP_Estudio')

## 2. Parametros generales

La simulacion cubre 2020-01 a 2025-12. Son 72 meses completos.


In [2]:
rng = Random(42)
periods = sim._periods()

periods[:3], periods[-3:], len(periods)


(['2020-01', '2020-02', '2020-03'], ['2025-10', '2025-11', '2025-12'], 72)

## 3. Jerarquia de clientes

Clientes simulados para Chile. Esta tabla representa la jerarquia de clientes:
industria, grupo economico, cliente y faena.


In [3]:
clients = sim._clients()
clientes_df = pd.DataFrame([sim._client_row(client) for client in clients])
clientes_df


,id_cliente,codigo_cliente_sap,nombre_cliente,pais,sociedad_sap,industria,grupo_economico,segmento,faena_operacion,region,estado_cliente
0,CLI-001,1000001001,Minera del Sol,Chile,CL01,Mineria,Grupo Minero Sol,Gran cuenta,Faena Norte,Antofagasta,Activo
1,CLI-002,1000001002,Minera Cordillera,Chile,CL01,Mineria,Grupo Cordillera,Gran cuenta,Faena Cordillera,Atacama,Activo
2,CLI-003,1000001003,Minera Pacifico,Chile,CL01,Mineria,Grupo Pacifico,Gran cuenta,Puerto Minero,Coquimbo,Activo
3,CLI-004,1000001004,Constructora Andina,Chile,CL01,Construccion,Grupo Andino,Cuenta estrategica,Obras RM,Metropolitana,Activo
4,CLI-005,1000001005,Infraestructura Sur,Chile,CL01,Construccion,Grupo InfraSur,Cuenta regional,Proyecto Ruta Sur,Biobio,Activo
5,CLI-006,1000001006,Energia Norte,Chile,CL01,Energia,Grupo Energia Norte,Cuenta estrategica,Planta Mejillones,Antofagasta,Activo
6,CLI-007,1000001007,Generacion Austral,Chile,CL01,Energia,Grupo Austral,Cuenta regional,Central Los Lagos,Los Lagos,Activo
7,CLI-008,1000001008,Puerto Industrial Chile,Chile,CL01,Portuario,Grupo Puerto Chile,Cuenta estrategica,Terminal San Antonio,Valparaiso,Activo
8,CLI-009,1000001009,Servicios Maritimos Valparaiso,Chile,CL01,Maritimo,Grupo Maritimo V,Cuenta regional,Puerto Valparaiso,Valparaiso,Activo
9,CLI-010,1000001010,Industria Celulosa BioBio,Chile,CL01,Forestal,Grupo Celulosa Sur,Cuenta regional,Planta BioBio,Biobio,Activo


## 4. Jerarquia de productos/equipos

Incluye grupos de equipos, subfamilias, estado comercial y ciclo de vida.


In [4]:
products = sim._products()
productos_df = pd.DataFrame([sim._product_row(product) for product in products])
productos_df


,id_producto_equipo,codigo_material_sap,familia_producto,subfamilia_producto,linea_negocio,tipo_equipo,modelo,comercializacion_equipo,estado_ciclo_vida,vida_util_esperada_meses
0,PROD-001,MAT-001001,Motores,Motores estaticos,Servicios,Motor,C175,Equipo comercializado hoy,En servicio,120
1,PROD-002,MAT-001002,Motores,Motores maritimos,Repuestos,Motor,3516,Equipo no comercializado / legado,Proximo a renovacion,144
2,PROD-003,MAT-002001,Generadores,Generadores diesel,Arriendos,Generador,XQ200,Equipo comercializado hoy,Vendido actual,120
3,PROD-004,MAT-002002,Generadores,Generadores respaldo,Servicios,Generador,C32,Equipo comercializado hoy,En servicio,120
4,PROD-005,MAT-003001,Soporte electrico,UPS,Servicios,UPS,UPS-IND,Equipo no comercializado / legado,Fin de vida util,96
5,PROD-006,MAT-003002,Soporte electrico,Tableros electricos,Repuestos,Tablero,TAB-MT,Equipo comercializado hoy,En servicio,96


## 5. Celdas cliente-producto

Cada celda cruza un cliente con un equipo/material. En estas celdas vive el RAPV.


In [5]:
cells = sim._cells(clients, products, rng)
celdas_df = pd.DataFrame(cells)

celdas_df.head(), celdas_df.shape


(   id_celda id_cliente id_producto_equipo codigo_cliente_sap  \
 0  CEL-0001    CLI-001           PROD-001         1000001001   
 1  CEL-0002    CLI-001           PROD-006         1000001001   
 2  CEL-0003    CLI-001           PROD-003         1000001001   
 3  CEL-0004    CLI-001           PROD-005         1000001001   
 4  CEL-0005    CLI-002           PROD-002         1000001002   
 
   codigo_material_sap   pais sociedad_sap industria   grupo_economico  \
 0          MAT-001001  Chile         CL01   Mineria  Grupo Minero Sol   
 1          MAT-003002  Chile         CL01   Mineria  Grupo Minero Sol   
 2          MAT-002001  Chile         CL01   Mineria  Grupo Minero Sol   
 3          MAT-003001  Chile         CL01   Mineria  Grupo Minero Sol   
 4          MAT-001002  Chile         CL01   Mineria  Grupo Cordillera   
 
       nombre_cliente   faena_operacion   familia_producto  \
 0     Minera del Sol       Faena Norte            Motores   
 1     Minera del Sol       Faena Nort

## 6. GAV mensual

Simula el gasto de administracion y ventas mensual. Luego se asigna
proporcionalmente a cada celda.


In [6]:
gav = sim._gav_monthly(periods)
gav_df = pd.DataFrame(gav)

gav_df.head(), gav_df.tail()


(   periodo   gav_total moneda                 fuente
 0  2020-01  1636991353    CLP  Simulado FI-CO / CeCo
 1  2020-02  1654701026    CLP  Simulado FI-CO / CeCo
 2  2020-03  1661184000    CLP  Simulado FI-CO / CeCo
 3  2020-04  1654703267    CLP  Simulado FI-CO / CeCo
 4  2020-05  1636995235    CLP  Simulado FI-CO / CeCo,
     periodo   gav_total moneda                 fuente
 67  2025-08  2181424130    CLP  Simulado FI-CO / CeCo
 68  2025-09  2172418468    CLP  Simulado FI-CO / CeCo
 69  2025-10  2181414792    CLP  Simulado FI-CO / CeCo
 70  2025-11  2206002684    CLP  Simulado FI-CO / CeCo
 71  2025-12  2239594217    CLP  Simulado FI-CO / CeCo)

## 7. Facturacion y margen post-venta

Esta es la tabla principal. Contiene ventas de equipos, repuestos, servicios,
arriendos, margenes, GAV asignado, RAPV y DOS.


In [7]:
facts = sim._facts(periods, cells, clients, products, gav, rng)
base_mensual_df = pd.DataFrame(facts)

base_mensual_df.head()


,periodo,anio,mes,id_celda,numero_factura_simulada,codigo_cliente_sap,nombre_cliente,industria,grupo_economico,faena_operacion,...,ventas_arriendos,margen_repuestos,margen_servicios,margen_arriendos,margen_postventa,gav_asignado,ratio_absorcion,dos,cobranza_alineada,moneda
0,2020-01,2020,1,CEL-0001,FAC-9000000001,1000001001,Minera del Sol,Mineria,Grupo Minero Sol,Faena Norte,...,5880122,11627111,8185660,2234446,22047217,47935443,0.4599,60,Si,CLP
1,2020-01,2020,1,CEL-0002,FAC-9000000002,1000001001,Minera del Sol,Mineria,Grupo Minero Sol,Faena Norte,...,3009762,5951379,4189859,1143709,11284948,53736879,0.2100,65,Si,CLP
2,2020-01,2020,1,CEL-0003,FAC-9000000003,1000001001,Minera del Sol,Mineria,Grupo Minero Sol,Faena Norte,...,14664433,8615123,6096552,5572484,20284159,51529377,0.3936,60,Si,CLP
3,2020-01,2020,1,CEL-0004,FAC-9000000004,1000001001,Minera del Sol,Mineria,Grupo Minero Sol,Faena Norte,...,5914162,12080521,8644699,2247381,22972601,46010268,0.4993,73,No,CLP
4,2020-01,2020,1,CEL-0005,FAC-9000000005,1000001002,Minera Cordillera,Mineria,Grupo Cordillera,Faena Cordillera,...,5588945,11303952,8049634,2123799,21477386,58471171,0.3673,70,No,CLP


In [8]:
relationship_cells = sim._relationship_cells(cells, facts)
celdas_relacion_df = pd.DataFrame(relationship_cells)

celdas_relacion_df.loc[
    :,
    [
        "id_celda",
        "nombre_cliente",
        "familia_producto",
        "subfamilia_producto",
        "facturacion_servicios_arriendos",
        "gav_mensual_prorrateado_familia",
        "margen_postventa_total",
        "rapv_periodo_total",
    ],
].head()


,id_celda,nombre_cliente,familia_producto,subfamilia_producto,facturacion_servicios_arriendos,gav_mensual_prorrateado_familia,margen_postventa_total,rapv_periodo_total
0,CEL-0001,Minera del Sol,Motores,Motores estaticos,1994530260,57387443,1292921260,0.3129
1,CEL-0002,Minera del Sol,Soporte electrico,Tableros electricos,1501821141,64332815,973605925,0.2102
2,CEL-0003,Minera del Sol,Generadores,Generadores diesel,2779084856,61690035,1588278533,0.3576
3,CEL-0004,Minera del Sol,Soporte electrico,UPS,2033292848,55082658,1311052321,0.3306
4,CEL-0005,Minera Cordillera,Motores,Motores maritimos,2481716979,70000624,1602728249,0.3180


## 8. Verificar calculo RAPV

RAPV = margen_postventa / gav_asignado.


In [9]:
check_cols = [
    "periodo",
    "nombre_cliente",
    "familia_producto",
    "margen_postventa",
    "gav_asignado",
    "ratio_absorcion",
    "dos",
]

sample = base_mensual_df.loc[:, check_cols].head(10).copy()
sample["rapv_recalculado"] = sample["margen_postventa"] / sample["gav_asignado"]
sample


,periodo,nombre_cliente,familia_producto,margen_postventa,gav_asignado,ratio_absorcion,dos,rapv_recalculado
0,2020-01,Minera del Sol,Motores,22047217,47935443,0.4599,60,0.459936
1,2020-01,Minera del Sol,Soporte electrico,11284948,53736879,0.2100,65,0.210004
2,2020-01,Minera del Sol,Generadores,20284159,51529377,0.3936,60,0.393643
3,2020-01,Minera del Sol,Soporte electrico,22972601,46010268,0.4993,73,0.499293
4,2020-01,Minera Cordillera,Motores,21477386,58471171,0.3673,70,0.367316
5,2020-01,Minera Cordillera,Motores,18511399,41512531,0.4459,74,0.445923
6,2020-01,Minera Pacifico,Soporte electrico,14243380,46461273,0.3066,63,0.306565
7,2020-01,Minera Pacifico,Soporte electrico,16671856,54372245,0.3066,69,0.306624
8,2020-01,Minera Pacifico,Motores,17276080,48357390,0.3573,68,0.357258
9,2020-01,Minera Pacifico,Generadores,27142642,55293569,0.4909,63,0.490882


## 9. Matriz cliente x grupo de equipos

Eje Y: jerarquia/agrupacion de clientes. Eje X: grupos de productos/equipos.
Cada celda muestra RAPV y DOS.


In [10]:
matrix = sim._matrix(cells, clients, products, facts)
matriz_df = pd.DataFrame(matrix)

matriz_df


,pais,industria,grupo_economico,codigo_cliente_sap,nombre_cliente,RAPV_Generadores,DOS_Generadores,RAPV_Motores,DOS_Motores,RAPV_Soporte electrico,DOS_Soporte electrico
0,Chile,Mineria,Grupo Minero Sol,1000001001,Minera del Sol,0.358,66.8,0.313,67.3,0.266,67.5
1,Chile,Mineria,Grupo Cordillera,1000001002,Minera Cordillera,,,0.343,66.7,,
2,Chile,Mineria,Grupo Pacifico,1000001003,Minera Pacifico,0.327,66.6,0.325,67.8,0.266,67.9
3,Chile,Construccion,Grupo Andino,1000001004,Constructora Andina,,,0.232,83.2,0.232,84.0
4,Chile,Construccion,Grupo InfraSur,1000001005,Infraestructura Sur,,,0.359,81.9,0.277,82.8
5,Chile,Energia,Grupo Energia Norte,1000001006,Energia Norte,0.38,70.4,0.439,69.9,0.337,70.0
6,Chile,Energia,Grupo Austral,1000001007,Generacion Austral,0.336,70.7,0.349,70.4,0.244,71.7
7,Chile,Portuario,Grupo Puerto Chile,1000001008,Puerto Industrial Chile,0.23,80.5,0.24,78.3,0.296,77.1
8,Chile,Maritimo,Grupo Maritimo V,1000001009,Servicios Maritimos Valparaiso,0.34,84.7,,,0.174,87.0
9,Chile,Forestal,Grupo Celulosa Sur,1000001010,Industria Celulosa BioBio,0.235,92.6,0.349,90.1,0.244,93.2


## 10. Resumen por grupo de productos

Permite ver que familias aportan mas al margen post-venta, RAPV y DOS.


In [11]:
family_summary = sim._family_summary(facts)
familia_df = pd.DataFrame(family_summary)

familia_df


,familia_producto,ventas_postventa,margen_postventa,gav_asignado,ratio_absorcion,dos_promedio,cobranza_alineada
0,Motores,44109884404,14225030016,45466453256,0.313,76.0,No
1,Soporte electrico,44054227095,14201450482,58734232087,0.242,77.1,No
2,Generadores,33091736134,10985979998,36903210470,0.298,77.0,No


## 11. Resumen por cliente

Permite identificar clientes con mayor aporte y revisar si el aporte esta
alineado con cobranza.


In [12]:
customer_summary = sim._customer_summary(facts)
cliente_resumen_df = pd.DataFrame(customer_summary)

cliente_resumen_df.head(12)


,nombre_cliente,ventas_postventa,margen_postventa,gav_asignado,ratio_absorcion,dos_promedio,cobranza_alineada
0,Minera Pacifico,16083000303,5230635081,17625967526,0.297,67.6,No
1,Minera del Sol,15880012057,5165858039,17171492498,0.301,67.3,No
2,Energia Norte,14995968814,4870753214,12290716244,0.396,70.1,No
3,Generacion Austral,12884024320,4189267212,14475464811,0.289,71.1,No
4,Puerto Industrial Chile,9617620671,3133353878,12495232034,0.251,78.6,No
5,Industria Celulosa BioBio,9158484017,2981971067,11070074315,0.269,92.0,No
6,Minera Cordillera,9159738206,2953506227,8618304441,0.343,66.7,No
7,Constructora Andina,8067730050,2601442053,11207169875,0.232,83.7,No
8,Agroindustrial Central,6940506002,2263251020,13318354273,0.170,99.7,No
9,Data Center Santiago,6625664447,2174049456,9050550249,0.240,62.6,Si


## 12. Control de calidad de datos

Validaciones simples antes de exportar.


In [13]:
quality = {
    "periodos": base_mensual_df["periodo"].nunique(),
    "clientes": base_mensual_df["codigo_cliente_sap"].nunique(),
    "familias_producto": base_mensual_df["familia_producto"].nunique(),
    "celdas": base_mensual_df["id_celda"].nunique(),
    "registros": len(base_mensual_df),
    "rapv_promedio_simple": base_mensual_df["ratio_absorcion"].mean(),
    "dos_promedio": base_mensual_df["dos"].mean(),
}

quality


{'periodos': 72,
 'clientes': 12,
 'familias_producto': 3,
 'celdas': 36,
 'registros': 2592,
 'rapv_promedio_simple': np.float64(0.29061338734567904),
 'dos_promedio': np.float64(76.69598765432099)}

## 13. Exportar Excel completo

Esta celda ejecuta el generador oficial y crea el Excel en `data/processed`
y OneDrive.


In [14]:
sim.main()


/Users/alejandroalvear/Library/CloudStorage/OneDrive-Finning/SAP_Estudio/resiliencia_financiera_tesis_mafi.xlsx
/Users/alejandroalvear/Documents/Alejandro/Desarrollo/Finanzas/MachineLearning/SAP_Estudio/data/processed/resiliencia_financiera_tesis_mafi.xlsx


## 14. Rutas de salida

Archivos generados por el script.


In [15]:
print("Local:", sim.LOCAL_FILE)
print("OneDrive:", sim.ONEDRIVE_FILE)


Local: /Users/alejandroalvear/Documents/Alejandro/Desarrollo/Finanzas/MachineLearning/SAP_Estudio/data/processed/resiliencia_financiera_tesis_mafi.xlsx
OneDrive: /Users/alejandroalvear/Library/CloudStorage/OneDrive-Finning/SAP_Estudio/resiliencia_financiera_tesis_mafi.xlsx
